# Edwards TIC auslesen

Der Kernel ist das Python, das die Notebook-Zellen ausführt. Klicke in VS Code oben rechts auf den Kernelnamen oder auf **Kernel auswählen**. Wähle bei Bedarf **Anderen Kernel auswählen → Python-Umgebungen** und dann **.venv** im Ordner `LAPIS-Control-Suite`. Unter Windows endet der zugehörige Pfad auf `LAPIS-Control-Suite\.venv\Scripts\python.exe`.

Für eine Messung am Aufbau setze in der Setup-Zelle `READ_HARDWARE = True` und führe sie mit **Umschalt+Enter** aus. Sie zeigt den TIC-Port aus der gewählten Datei an (aktuell COM13). Führe danach nur **Druck ablesen** aus. Jede weitere Zelle ist eine optionale Einzelabfrage und kann direkt nach dem Setup gestartet werden.

In [3]:
from contextlib import contextmanager
from pathlib import Path
import sys

print(f"Notebook-Python: {sys.executable}")
roots = [Path.cwd(), *Path.cwd().parents, Path.cwd() / "LAPIS-Control-Suite"]
project = next((p for p in roots if (p / "src/lapis_control").is_dir()
                and (p / "config/devices.example.json").is_file()), None)
if project is None:
    raise FileNotFoundError("Öffne das Notebook im LAPIS-Projektordner.")
src = str(project / "src")
if src not in sys.path:
    sys.path.insert(0, src)

try:
    from serial import Serial
except ImportError as exc:
    raise ImportError(
        f"Diesem Kernel fehlt die pyserial-Schnittstelle. Wähle oben rechts "
        f"die Python-Umgebung {project / '.venv'} und starte diese Zelle erneut."
    ) from exc

from lapis_control.configuration import load_devices
from lapis_control.devices.vacuum.edwards_tic import EdwardsTIC, EdwardsTicError

READ_HARDWARE = False
CONFIG_PATH = project / "config/devices.example.json"
tic = load_devices(CONFIG_PATH)["vacuum"]
assert isinstance(tic, EdwardsTIC)
print(f"{CONFIG_PATH.name} · TIC-Port {tic.port}")

@contextmanager
def diagnostic(label):
    try:
        yield
    except EdwardsTicError as exc:
        print(f"{label} · Abfrage fehlgeschlagen: {exc}")


def show_measurement(label, read):
    with diagnostic(label):
        result = read()
        print(f"{label}: {result['value']:g} {result['unit']} · "
              f"Alert {result['alert_id']} ({result['alert']}) · Priorität {result['priority_code']}")

Notebook-Python: c:\Users\jvelte01\Desktop\LAPIS-Control Suite\LAPIS-Control-Suite\.venv\Scripts\python.exe
devices.example.json · TIC-Port COM13


## Druck ablesen

Wähle den Kanal der angeschlossenen Messröhre (1, 2 oder 3). Bei ausgeschalteter oder gestörter Messröhre erscheint die Diagnose. Die folgende Kanalübersicht hilft bei der Zuordnung.

In [4]:
GAUGE_CHANNEL = 1
if READ_HARDWARE:
    with tic:
        pressure_pa = tic.read_pressure_pa(GAUGE_CHANNEL)
    print(f"Druck an Kanal {GAUGE_CHANNEL}: {pressure_pa:.4g} Pa · {pressure_pa / 100:.4g} mbar")
else:
    print("Für eine Messung READ_HARDWARE im Setup aktivieren und die Setup-Zelle erneut ausführen.")

Druck an Kanal 1: 2.283e-06 Pa · 2.283e-08 mbar


## Messröhren und Meldungen

Fragt die drei Eingänge einzeln ab. Rohwert, Einheit und Zustand helfen etwa bei einem unbelegten Anschluss oder einer als Spannung konfigurierten Messröhre.

In [5]:
if READ_HARDWARE:
    with tic:
        for channel in (1, 2, 3):
            with diagnostic(f"Kanal {channel}"):
                gauge = tic.read_pressure(channel)
                print(f"Kanal {channel}: {gauge['value']:g} {gauge['unit']} (Rohwert) · "
                      f"Zustand {gauge['state_code']} ({gauge['state']}) · "
                      f"Alert {gauge['alert_id']} ({gauge['alert']}) · Priorität {gauge['priority']}")

Kanal 1: 2.283e-06 Pa (Rohwert) · Zustand 11 (On) · Alert 0 (No Alert) · Priorität 0
Kanal 2: 9.9e+09 Pa (Rohwert) · Zustand 0 (Not connected) · Alert 0 (No Alert) · Priorität 0
Kanal 3: 9.9e+09 Pa (Rohwert) · Zustand 0 (Not connected) · Alert 0 (No Alert) · Priorität 0


## Pumpenzustände am TIC

Je Pumpe erscheinen Zustandsnummer, Meldung, Priorität und Rohantwort. Ein Fehlerzustand kann auch bei Alert-ID 0 vorliegen.

In [6]:
if READ_HARDWARE:
    with tic:
        print("nEXT 300:", tic.read_turbo_pump_state())
        print("nXDS:", tic.read_backing_pump_state())

nEXT 300: {'object_id': 904, 'state_code': 6, 'state': 'Fault Braking', 'alert_id': 0, 'alert': 'No Alert', 'priority_code': 0, 'priority': 'OK', 'raw_payload': '6;0;0', 'is_fault_braking': True, 'has_fault': True}
nXDS: {'object_id': 910, 'state_code': 5, 'state': 'Unknown (5)', 'alert_id': 0, 'alert': 'No Alert', 'priority_code': 0, 'priority': 'OK', 'raw_payload': '5;0;0'}


## Fehlermeldungen direkt aus den Pumpen

Liest die Diagnose der nEXT und die Warn- und Fehlerregister der nXDS über den TIC. Alle gesetzten Fehlerbits werden angezeigt, auch wenn eine Meldung am Aufbau bekannt ist. Eine fehlende Antwort wird für die jeweilige Pumpe ausgewiesen.

In [7]:
if READ_HARDWARE:
    with tic:
        with diagnostic("nEXT 300"):
            turbo = tic.read_next_speed_and_status()
            print(f"nEXT 300 · Statuswort 0x{turbo['status_word_hex']}")
            print("Fehlerbits:", turbo['active_faults'] or "keine gesetzt")
        with diagnostic("nXDS"):
            backing = tic.read_nxds_status()
            print(f"nXDS · Rohantwort {backing['raw_payload']}")
            print("Warnbits:", backing['warnings'] or "keine gesetzt")
            print("Fehlerbits:", backing['faults'] or "keine gesetzt")

nEXT 300 · Statuswort 0x000502BC
Fehlerbits: keine gesetzt
nXDS · Rohantwort 30;0472;0500;0000;0000
Warnbits: keine gesetzt
Fehlerbits: keine gesetzt


## Drehzahl und Leistung

Zeigt die gemessene TMP-Drehzahl und die vom TIC verfügbaren Drehzahl- und Leistungswerte beider Pumpen. Die Prozentwerte beziehen sich auf die jeweilige Pumpe.

In [9]:
if READ_HARDWARE:
    with tic:
        with diagnostic("TMP-Drehzahl"):
            speed = tic.read_next_speed_and_status()
            print(f"TMP-Drehzahl: {speed['speed_rpm']} U/min · {speed['speed_rps']} U/s")
        show_measurement("TMP-Drehzahl am TIC", tic.read_turbo_speed_percent)
        show_measurement("TMP-Leistung", tic.read_turbo_power)
        show_measurement("Vorpumpendrehzahl am TIC", tic.read_backing_speed)
        show_measurement("Vorpumpenleistung", tic.read_backing_power)

TMP-Drehzahl: 59940 U/min · 999 U/s
TMP-Drehzahl am TIC: 99.9 % · Alert 0 (No Alert) · Priorität 0
TMP-Leistung: 15.4 W · Alert 0 (No Alert) · Priorität 0
Vorpumpendrehzahl am TIC: 100 % · Alert 0 (No Alert) · Priorität 0
Vorpumpenleistung: 222 W · Alert 0 (No Alert) · Priorität 0


## Temperaturen

Liest TIC-Netzteil, TIC-Innentemperatur sowie die Sensoren der beiden Pumpen aus. Welche nXDS-Sensoren Werte liefern, hängt von der Geräteausführung ab.

In [10]:
if READ_HARDWARE:
    with tic:
        show_measurement("TIC-Netzteil", tic.read_ps_temperature)
        show_measurement("TIC innen", tic.read_internal_temperature)
        with diagnostic("nEXT 300"):
            temps = tic.read_next_temperatures()
            print(f"TMP-Motor: {temps['motor_temperature_c']} °C")
            print(f"TMP-Controller: {temps['controller_temperature_c']} °C")
        with diagnostic("nXDS"):
            temps = tic.read_nxds_temperatures()
            for label, key in (("nXDS-Pumpe", "pump_temperature_c"),
                               ("nXDS-Controller", "controller_temperature_c")):
                value = temps[key]
                print(f"{label}: {value} °C" if value is not None else f"{label}: Sensor ungenutzt")

TIC-Netzteil: 36.3 °C · Alert 0 (No Alert) · Priorität 0
TIC innen: 28.1 °C · Alert 0 (No Alert) · Priorität 0
TMP-Motor: 29 °C
TMP-Controller: 26 °C
nXDS-Pumpe: 22 °C
nXDS-Controller: 48 °C
